# 🚀 Takatsuki SLM Pre-Training & Alignment Studio (Kaggle Dual-T4 GPU)

Welcome to the **Takatsuki AI Lab** training suite optimized for **Kaggle's Free Tier (2x NVIDIA T4 GPUs / 30 hrs/week)**.

### What this notebook does:
1. **GPU Environment Setup:** Configures PyTorch with FP16 Automatic Mixed Precision (AMP) & Flash Scaled Dot-Product Attention (SDPA).
2. **Corpus & Tokenizer Streaming:** Prepares the 32k Byte-Level BPE Takatsuki Tokenizer.
3. **Model Selection:** Supports training **Takatsuki-150M** (scratch) or fine-tuning **Takatsuki-1B / 3B**.
4. **High-Speed GPU Training:** Achieves **~15,000 - 22,000 tokens/sec** (finishes in ~16 hrs vs 33 days on CPU).
5. **Live Generation:** Interactive chat with your newly trained model directly inside the notebook.

In [ ]:
# Step 1: Verify Kaggle GPU Environment & Install Dependencies
!nvidia-smi
!pip install -q --upgrade pip
!pip install -q torch transformers tokenizers datasets accelerate einops pyyaml tqdm matplotlib

import os, sys, math, time, json
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.optim import AdamW
import matplotlib.pyplot as plt

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB VRAM)")

In [ ]:
# Step 2: Clone or Pull the SLM-Lab Repository
!rm -rf slm-lab
!git clone https://github.com/neverlone/slm-lab.git
%cd slm-lab
sys.path.append(os.getcwd())
print("Repository loaded successfully!")

In [ ]:
# Step 3: Model Architecture Configuration (150M Parameter Model)
from src.model.transformer import SLMForCausalLM, ModelArgs

MODEL_CONFIG_150M = {
    "dim": 768,
    "n_layers": 12,
    "n_heads": 12,
    "n_kv_heads": 4,
    "vocab_size": 32768,
    "max_seq_len": 2048,
    "rope_theta": 10000.0,
    "norm_eps": 1e-5,
    "tie_word_embeddings": True
}

args = ModelArgs(**MODEL_CONFIG_150M)
model = SLMForCausalLM(args)
print(f"Initialized Takatsuki-150M with {model.count_parameters():,} trainable parameters.")

In [ ]:
# Step 4: Stream Curated Corpus & Train Takatsuki 32k BPE Tokenizer
!python scripts/prepare_takatsuki_data.py --samples_per_source 25000

In [ ]:
# Step 5: GPU Accelerated Pre-training Engine (Mixed Precision FP16 + Gradient Accumulation)
import yaml
from src.dataset.dataset import create_dataloader

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Hyperparameters tuned for Kaggle GPU
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2  # Effective batch size = 32 (65,536 tokens per optimizer step)
MAX_LR = 6e-4
MIN_LR = 6e-5
WARMUP_STEPS = 500
MAX_STEPS = 20000
SAVE_INTERVAL = 1000

optimizer = AdamW(model.parameters(), lr=MAX_LR, betas=(0.9, 0.95), weight_decay=0.1)
scaler = GradScaler()  # FP16 gradient scaling

data_path = "data/tokenized/train.bin"
dataloader = create_dataloader(bin_path=data_path, batch_size=BATCH_SIZE, seq_len=args.max_seq_len)
data_iter = iter(dataloader)

def get_lr(it, warmup_iters=WARMUP_STEPS, lr_decay_iters=MAX_STEPS, max_lr=MAX_LR, min_lr=MIN_LR):
    if it < warmup_iters:
        return max_lr * (it + 1) / (warmup_iters + 1)
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

os.makedirs("checkpoints/kaggle", exist_ok=True)
loss_history = []
t0 = time.time()

print("=" * 65)
print("🚀 STARTING TAKATSUKI-150M GPU TRAINING RUN (FP16)")
print("=" * 65)

model.train()
for step in range(MAX_STEPS):
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0

    for _ in range(GRAD_ACCUM_STEPS):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            x, y = next(data_iter)

        x, y = x.to(device), y.to(device)

        with autocast(dtype=torch.float16):
            _, loss, _ = model(x, labels=y)
            loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        accum_loss += loss.item() * GRAD_ACCUM_STEPS

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    loss_history.append(accum_loss)

    if step % 20 == 0:
        dt = time.time() - t0
        t0 = time.time()
        tok_processed = BATCH_SIZE * GRAD_ACCUM_STEPS * args.max_seq_len * 20
        tok_per_sec = tok_processed / max(dt, 1e-4)
        pct = (step / MAX_STEPS) * 100
        print(f"[Step {step:05d}/{MAX_STEPS:05d} ({pct:4.1f}%)] Loss: {accum_loss:.4f} | LR: {lr:.2e} | Speed: {tok_per_sec:,.0f} tok/s")

    if step > 0 and (step % SAVE_INTERVAL == 0 or step == MAX_STEPS - 1):
        ckpt_file = f"checkpoints/kaggle/takatsuki_step_{step}.pt"
        torch.save({
            "step": step,
            "model_state_dict": model.state_dict(),
            "loss": accum_loss,
            "args": args
        }, ckpt_file)
        print(f"--> Checkpoint saved: {ckpt_file}")

print("\n✅ Training Finished!")

In [ ]:
# Step 6: Plot Loss Convergence
plt.figure(figsize=(10, 4))
plt.plot(loss_history, label="Training Loss", color="#6366f1", alpha=0.8)
plt.title("Takatsuki-150M GPU Loss Convergence")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.show()

In [ ]:
# Step 7: Live Generation & Chat Evaluation
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast(tokenizer_file="data/tokenizer/takatsuki_tokenizer.json")
model.eval()

prompt = "<|im_start|>user\nWhat is the essence of courage in one sentence?<|im_end|>\n<|im_start|>assistant\n"
input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)

with torch.no_grad():
    output_ids = model.generate(input_ids, max_new_tokens=64, temperature=0.7, top_p=0.9)

generated_text = tokenizer.decode(output_ids[0].tolist())
print("=== GENERATED RESPONSE ===")
print(generated_text)